# E3: piloto de UNet-AT, dos épocas por brazo

Selecciona un entorno GPU y ejecuta todas las celdas. Este cuaderno termina después de dos épocas
de `dbnet` y dos de `dbnet_at`, semilla 11, usando train y validación completos. No lee imágenes de test.
No lanza las otras semillas ni la búsqueda de hiperparámetros.

Necesita **paquete_colab_e2_v2.zip**, ya existente, en MyDrive o en el panel de archivos.
No necesita los pesos compactos: el original se entrena desde cero.
La carpeta de salida es NUEVA: `/content/drive/MyDrive/tesis_imo_e3_original_piloto_v3`.
Es un destino que este cuaderno crea, no una carpeta recuperada de Daniel.

La pausa no cambia el máximo de 30 épocas ni la paciencia 5 de la receta. Guarda `last.pt` y
el mejor checkpoint en Drive. La reanudación requiere el mismo código, configuración y entorno.
El tiempo y las unidades de procesamiento solo se podrán estimar después de esta medición GPU.

**Estado de revisión:** hay una discrepancia sin resolver entre el oráculo geométrico histórico y
el código recuperado (rombo de 10 px, brazo DBNet: 0,9730 frente a 0,9823). Este cuaderno es un piloto
técnico de tiempo y reanudación; no demuestra superioridad ni autoriza considerar cerrado el protocolo.
La campaña confirmatoria debe esperar a que esa discrepancia se explique y se cierre la auditoría.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import subprocess, sys, os, json, hashlib, zipfile
subprocess.run(['nvidia-smi'], check=True)
subprocess.run(['mountpoint', '-q', '/content/drive'], check=True)
OUTPUT = Path('/content/drive/MyDrive/tesis_imo_e3_original_piloto_v3')
OUTPUT.mkdir(parents=True, exist_ok=True)


In [ ]:
EXPECTED_ZIP_SHA256 = '954f4fe5663d9fedd676dcee4f1f7fb9086a2a7f5c1907ce902b5df4c109cb49'
ZIP = next((p for p in [Path('/content/paquete_colab_e2_v2.zip'), Path('/content/drive/MyDrive/paquete_colab_e2_v2.zip')] if p.exists()), None)
if ZIP is None: raise FileNotFoundError('Sube paquete_colab_e2_v2.zip al panel de archivos y repite esta celda.')
def sha_file(p):
    h=hashlib.sha256()
    with p.open('rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''): h.update(b)
    return h.hexdigest()
assert sha_file(ZIP)==EXPECTED_ZIP_SHA256, 'ZIP incorrecto o modificado; usa el archivo v2 entregado con este notebook'
BASE=Path('/content/imo_e2_v2')
BASE.mkdir(exist_ok=True)
with zipfile.ZipFile(ZIP) as z:
    for n in z.namelist():
        if not (BASE/n).resolve().is_relative_to(BASE): raise ValueError('Ruta no válida en ZIP')
    z.extractall(BASE)
manifest=json.loads((BASE/'PACKAGE_MANIFEST.json').read_text())
for name, expected in manifest['files'].items():
    assert sha_file(BASE/name)==expected, name
PROJECT=BASE/'tesis_imo_para_codex'
print('Paquete íntegro:',len(manifest['files']),'archivos')


## Entorno aislado y fijado
Se crea Python 3.11 fuera del kernel de Colab. La instalación puede tardar varios minutos; no cambia el Python del notebook. No se compilan extensiones CUDA: se usan ruedas publicadas por sus autores para torch 2.5/CUDA 12.


In [ ]:
subprocess.run([sys.executable,'-m','pip','install','uv==0.6.9'],check=True)
ENV=Path('/content/imo-e2-env')
if not (ENV/'bin/python').exists():
    subprocess.run([sys.executable,'-m','uv','venv','--python','3.11',str(ENV)],check=True)
PY=str(ENV/'bin/python')
def pip_install(*items):
    subprocess.run([sys.executable,'-m','uv','pip','install','--python',PY,*items],check=True)
pip_install('torch==2.5.1','torchvision==0.20.1','--index-url','https://download.pytorch.org/whl/cu121')
pip_install('-r',str(PROJECT/'requirements-e2.txt'))
subprocess.run([PY,'-c','import torch, torchvision; assert torch.cuda.is_available(); print(torch.__version__,torch.cuda.get_device_name(0))'],check=True)
freeze=subprocess.check_output([sys.executable,'-m','uv','pip','freeze','--python',PY],text=True)
(OUTPUT/'environment.freeze.txt').write_text(freeze)


In [ ]:
pip_install('opencv-python-headless==4.10.0.84')


In [ ]:
E3 = Path('/content/e3_piloto')
ARCHIVOS = {'src/__init__.py': '', 'src/geometria_at.py': '"""Geometria de DBNet-AT: etiquetas (C1), expansion consistente (C2) y posproceso. Modulo autonomo.\n\nFuentes:\n- DBNet: Liao, Wan, Yao, Chen y Bai (2020), "Real-time Scene Text Detection with Differentiable Binarization",\n  AAAI. De ahi salen el encogimiento D = A(1 - r^2)/L con r = 0,4, el mapa de umbral y la expansion D\' = A\' r\'/L\'\n  con r\' = 1,5. Los valores 0,3 (binarizacion) y 0,5 (puntaje minimo de region) son los del codigo oficial.\n- Formulacion de DBNet-AT: entregable_3/diseno.md, seccion 3 (C1 en 3.2, C2 en 3.3, teorema en 3.5).\n\nNotacion (la del teorema de 3.5):\n- K es una region (poligono convexo si se quiere la garantia exacta). A, L y g son su area, su perimetro y su\n  radio inscrito (la mayor distancia de un pixel de K al fondo).\n- r = 0,4 es la razon de encogimiento de DBNet; g0 = 4 px es el paso del mapa mas profundo de la U-Net compacta\n  (1/4 de 256), fijado por la arquitectura.\n- phi(g) = g^2 / (g^2 + g0^2) es la atenuacion por grosor de C1. Con g0 = 0, phi = 1 y se recupera DBNet.\n- La regla de encogimiento es rho(K) = phi(g) (1 - r^2) A / L. El nucleo es K_{-D} = {x en K : dist(x, borde) >= D}\n  con D = rho(K).\n- El desplazamiento en inglete de un poligono P = interseccion de {n_j . x <= c_j} es\n  M(P, d) = interseccion de {n_j . x <= c_j + d}: se mueve cada lado d hacia afuera y se cortan los consecutivos.\n- C2 busca el d que cumple d = rho(M(K\', d)), con K\' el nucleo predicho. Teorema: si K es convexo, ningun lado\n  colapsa al encogerse (S1) y sus angulos interiores son >= 60 grados (S2), la solucion es unica, d = D, y\n  M(K\', D) = K.\n\nCambio respecto a la version de entregable_3/codigo/etiquetas_at.py: el desplazamiento en inglete ya no lo hace\npyclipper (algoritmo de Vatti) sino el codigo de este archivo (inglete_poligono / rellenar_inglete). pyclipper\nqueda solo como prueba cruzada en tests/test_original_model.py.\n\nTodo lo demas viene de E2: instancias por componente conexo (vecindad 8), perimetro con cv2.arcLength, encoger\ny expandir con la transformada de distancia, umbrales del mapa 0,3-0,7, binarizacion 0,3, puntaje minimo 0,5 y\narea minima 9 px.\n"""\nimport cv2\nimport numpy as np\nfrom scipy import ndimage\n\n# ----------------------------------------------------------------------------- constantes\nG0 = 4.0            # px: paso del mapa mas profundo de la U-Net compacta (e3, 1/4 de 256)\nR = 0.4             # razon de encogimiento de DBNet\nR_PRIMA = 1.5       # razon de expansion fija de DBNet\nLIMITE_INGLETE = 2.0  # limite de inglete (multiplos de d). Una esquina de 90 grados necesita raiz(2)\nESCALA = 16         # las coordenadas del relleno van en 1/16 de pixel (cv2.fillPoly con shift = 4)\nAJUSTE_PX = 0.75    # corrimiento de discretizacion del contorno (calibrar_ajuste.py, 200 imagenes de train)\nOCHO = np.ones((3, 3), bool)\n\n\n# ----------------------------------------------------------------------------- primitivas de E2 (reescritas)\ndef instancias(mascara):\n    """Cada componente conexo (vecindad 8) de la mascara es una instancia. Devuelve (etiquetas, n)."""\n    return ndimage.label(mascara > 0, structure=OCHO)\n\n\ndef area_perimetro(inst):\n    """A = numero de pixeles; L = longitud del contorno externo (cv2.arcLength). L >= 1 para no dividir por 0."""\n    contornos, _ = cv2.findContours(inst.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)\n    perim = sum(cv2.arcLength(c, True) for c in contornos)\n    return float(inst.sum()), max(float(perim), 1.0)\n\n\ndef encoger(inst, d):\n    """Nucleo: los pixeles a distancia > d del fondo (equivale al desplazamiento de Vatti con esquinas vivas\n    en poligonos convexos, Lema 1 de diseno.md 3.5)."""\n    return ndimage.distance_transform_edt(inst) > d\n\n\ndef expandir(inst, d):\n    """Expansion euclidea: se anaden los pixeles de fondo a distancia <= d (esquinas redondeadas, como DBNet)."""\n    return ndimage.distance_transform_edt(~inst) <= d\n\n\ndef componentes(binaria, area_min):\n    """Etiqueta los componentes de una mascara y devuelve (etiquetas, lista de etiquetas con area >= area_min)."""\n    etq, n = instancias(binaria)\n    if n == 0:\n        return etq, []\n    areas = ndimage.sum(np.ones_like(etq), etq, index=np.arange(1, n + 1))\n    return etq, [i + 1 for i, a in enumerate(areas) if a >= area_min]\n\n\n# ----------------------------------------------------------------------------- C1: la regla de encogimiento\ndef phi(g, g0=G0):\n    """Atenuacion por grosor de C1: g^2 / (g^2 + g0^2). Con g0 <= 0 vale 1 (C1 apagado, regla de DBNet)."""\n    return 1.0 if g0 <= 0 else g * g / (g * g + g0 * g0)\n\n\ndef regla(region, r=R, g0=G0):\n    """rho(K) = phi(g) (1 - r^2) A / L: distancia de encogimiento de una region (bool)."""\n    A, L = area_perimetro(region)\n    g = float(ndimage.distance_transform_edt(region).max()) if region.any() else 0.0\n    return phi(g, g0) * A * (1 - r ** 2) / L\n\n\ndef at_objetivos(y, v, r=R, g0=G0, tmin=0.3, tmax=0.7):\n    """Etiquetas de entrenamiento (las de DBNet con D = rho(K) en lugar de A(1 - r^2)/L).\n\n    y: mascara binaria (H, W) de rotulos; v: zona valida (bool). Devuelve, con un eje de canal delante:\n    db_gt (nucleo, el blanco de P y de B), db_mask (donde cuenta la perdida), db_umbral (blanco de T, entre tmin y\n    tmax) y db_umbral_mask (la banda donde cuenta la perdida de T). La banda de T usa el mismo D que el nucleo.\n    Una instancia cuyo nucleo desaparece se ignora (db_mask = 0), como el codigo oficial de DBNet.\n    Con g0 = 0 es exactamente la etiqueta de DBNet."""\n    etq, n = instancias(y)\n    gt = np.zeros(y.shape, np.float32)\n    mask = v.astype(np.float32).copy()\n    umbral = np.zeros(y.shape, np.float32)\n    umbral_mask = np.zeros(y.shape, np.float32)\n    for k in range(1, n + 1):\n        inst = etq == k\n        D = regla(inst, r, g0)\n        s = encoger(inst, D)\n        if not s.any():\n            mask[inst] = 0\n            continue\n        gt[s] = 1\n        gd = expandir(inst, D)\n        dist = np.where(inst, ndimage.distance_transform_edt(inst), ndimage.distance_transform_edt(~inst))\n        t = np.clip(1 - dist / max(D, 1e-6), 0, 1)  # 1 en el borde de la region, 0 a distancia D\n        umbral[gd] = np.maximum(umbral[gd], t[gd])\n        umbral_mask[gd] = 1\n    umbral = umbral * (tmax - tmin) + tmin\n    return {"db_gt": gt[None], "db_mask": mask[None], "db_umbral": umbral[None], "db_umbral_mask": umbral_mask[None]}\n\n\n# ----------------------------------------------------------------------------- inglete propio (reemplaza a pyclipper)\ndef _orientar(p):\n    """Poligono (n, 2) float sin vertices repetidos y con area con signo positiva (orden antihorario en el\n    plano x-y; en coordenadas de imagen, con y hacia abajo, es el horario visual: no importa, la aritmetica es la\n    misma). Con esa orientacion la normal exterior del lado (dx, dy) es (dy, -dx)/|e|."""\n    p = np.asarray(p, float)\n    q = p[np.any(np.abs(p - np.roll(p, 1, axis=0)) > 1e-9, axis=1)]  # quita vertices consecutivos repetidos\n    area2 = np.sum(q[:, 0] * np.roll(q[:, 1], -1) - np.roll(q[:, 0], -1) * q[:, 1])\n    return q if area2 >= 0 else q[::-1]\n\n\ndef _geometria_lados(p):\n    """Vectores unitarios t_i y normales exteriores n_i de los lados i = p[i] -> p[i+1] de un poligono orientado."""\n    e = np.roll(p, -1, axis=0) - p\n    t = e / np.linalg.norm(e, axis=1, keepdims=True)\n    n = np.stack([t[:, 1], -t[:, 0]], axis=1)\n    return t, n\n\n\ndef _esquina(v, t1, n1, t2, n2, d, limite):\n    """Puntos de la esquina desplazada en el vertice v, entre el lado entrante (t1, n1) y el saliente (t2, n2).\n\n    Inglete: la interseccion de las dos rectas desplazadas, m = v + d (n1 + n2) / (1 + n1 . n2). Esta a una\n    distancia d / sen(theta/2) de v (theta = angulo interior). Si esa distancia pasa de limite * d, la punta se\n    corta con una recta perpendicular a la bisectriz, a limite * d de v (Clipper la llama \'squaring\'); entonces\n    salen dos puntos, uno sobre cada recta desplazada. Con limite = 2 el corte aparece bajo 60 grados (supuesto S2\n    del teorema). Devuelve una lista de 1 o 2 puntos."""\n    den = 1.0 + float(n1 @ n2)\n    if den > 1e-12 and 1.0 / np.sqrt(den / 2.0) <= limite:\n        return [v + d * (n1 + n2) / den]\n    u = n1 + n2\n    nu = np.linalg.norm(u)\n    u = u / nu if nu > 1e-12 else t1  # giro de 180 grados: se corta perpendicular al lado entrante\n    L = limite * d\n    pts = []\n    for t, n in ((t1, n1), (t2, n2)):\n        ct = float(t @ u)\n        if abs(ct) < 1e-12:\n            pts.append(v + d * n)\n        else:\n            pts.append(v + d * n + ((L - d * float(n @ u)) / ct) * t)\n    return pts\n\n\ndef inglete_poligono(p, d, limite=LIMITE_INGLETE):\n    """Desplazamiento en inglete M(P, d) de un poligono CONVEXO p (n, 2): cada lado d hacia afuera y se cortan los\n    consecutivos (diseno.md 3.5). Devuelve el poligono desplazado (m, 2); m >= n si hay esquinas cortadas.\n\n    Solo es correcto para poligonos convexos y d >= 0. Con un poligono no convexo, las esquinas entrantes\n    cruzarian los lados: para eso esta rellenar_inglete, que usa piezas convexas."""\n    p = _orientar(p)\n    t, n = _geometria_lados(p)\n    salida = []\n    for i in range(len(p)):\n        j = i - 1  # lado entrante al vertice i\n        salida.extend(_esquina(p[i], t[j], n[j], t[i], n[i], d, limite))\n    return np.asarray(salida)\n\n\ndef _piezas_inglete(p, d, limite=LIMITE_INGLETE):\n    """Piezas convexas cuya union es M(P, d) para CUALQUIER poligono simple (convexo o no): el poligono mismo, un\n    rectangulo sobre cada lado (de p[i], p[i+1] a d hacia afuera) y una cuna en cada vertice convexo (la esquina de\n    _esquina). En un vertice entrante no se pone cuna: los rectangulos de sus dos lados ya se solapan alli.\n    Caso limite documentado: el desplazamiento por union de piezas no recorta entrantes mas cortos que d; es el\n    desplazamiento \'exacto\' del conjunto, no el contorno que daria un recorte de bucles como el de Vatti."""\n    p = _orientar(p)\n    t, n = _geometria_lados(p)\n    k = len(p)\n    piezas = [p]\n    for i in range(k):\n        a, b = p[i], p[(i + 1) % k]\n        piezas.append(np.array([a, b, b + d * n[i], a + d * n[i]]))\n        j = i - 1\n        if t[j][0] * t[i][1] - t[j][1] * t[i][0] > 1e-12:  # vertice convexo (giro hacia el interior)\n            punta = _esquina(p[i], t[j], n[j], t[i], n[i], d, limite)\n            piezas.append(np.array([p[i], p[i] + d * n[j], *punta, p[i] + d * n[i]]))\n    return piezas\n\n\ndef _pintar(salida, poligono):\n    """Rellena un poligono float en la mascara uint8 (coordenadas en 1/16 de pixel, sin suavizado)."""\n    pts = np.round(np.asarray(poligono) * ESCALA).astype(np.int32)\n    cv2.fillPoly(salida, [pts], 1, lineType=cv2.LINE_8, shift=4)\n\n\ndef _contornos(c):\n    """Contornos externos del nucleo c (bool), simplificados con Douglas-Peucker a 1 px para que la escalera de\n    pixeles de un borde inclinado no genere picos al unir en inglete."""\n    cont, _ = cv2.findContours(c.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)\n    return [(cv2.approxPolyDP(k, 1.0, True) if len(k) > 4 else k)[:, 0, :].astype(float) for k in cont]\n\n\ndef rellenar_inglete(forma, contornos, d, limite=LIMITE_INGLETE):\n    """Mascara uint8 (forma) con el desplazamiento en inglete, d hacia afuera, de los contornos dados.\n    Contorno convexo: un solo poligono (inglete_poligono). No convexo: union de piezas (_piezas_inglete)."""\n    salida = np.zeros(forma, np.uint8)\n    for p in contornos:\n        if len(p) < 3:\n            continue\n        if cv2.isContourConvex(np.round(p).astype(np.int32)[:, None, :]):\n            _pintar(salida, inglete_poligono(p, d, limite))\n        else:\n            for pieza in _piezas_inglete(p, d, limite):\n                _pintar(salida, pieza)\n    return salida\n\n\ndef expandir_inglete(c, d, ajuste=None, limite=LIMITE_INGLETE):\n    """Desplaza hacia afuera el contorno del nucleo c en d px con uniones en inglete (esquinas vivas) y rellena.\n\n    El contorno de cv2 pasa por los centros de los pixeles del borde y el relleno no coincide exactamente con el\n    conjunto de pixeles de la etiqueta: se desplaza d + ajuste (AJUSTE_PX, un corrimiento de discretizacion que no\n    depende del tamano y se calibro solo con imagenes de train). Devuelve una mascara bool que incluye a c."""\n    desp = d + (AJUSTE_PX if ajuste is None else ajuste)\n    if desp <= 0:\n        return c.copy()\n    salida = rellenar_inglete(c.shape, _contornos(c), desp, limite)\n    return (salida > 0) | c\n\n\n# ----------------------------------------------------------------------------- C2: expansion consistente\ndef expansion_consistente(comp, v, r=R, g0=G0, iteraciones=16, inglete=True, ajuste=None):\n    """d tal que regla(expandir(comp, d) & v) = d, por biseccion; devuelve la region expandida (bool, tamano\n    completo). Es la inversa de C1: M(K\', d) con d = rho(M(K\', d)) (diseno.md 3.3). La region se recorta a su caja\n    con un margen suficiente para que los calculos sean locales.\n    inglete=True expande con esquinas vivas; False, por distancia euclidea (esquinas redondas)."""\n    ys, xs = np.nonzero(comp)\n    g_k = float(ndimage.distance_transform_edt(comp).max())\n    alto = 6.0 * (g_k + 1.0) + 2.0  # cota: en una franja larga d ~ 5,25 g\'; en una compacta d ~ 0,72 g\'\n    m = int(np.ceil(1.5 * alto)) + 2  # el inglete a 90 grados sale raiz(2) d por la esquina\n    y0, y1 = max(ys.min() - m, 0), min(ys.max() + m + 1, comp.shape[0])\n    x0, x1 = max(xs.min() - m, 0), min(xs.max() + m + 1, comp.shape[1])\n    c, vv = comp[y0:y1, x0:x1], v[y0:y1, x0:x1]\n    fuera = None if inglete else ndimage.distance_transform_edt(~c)\n\n    def expandida(d):\n        return (expandir_inglete(c, d, ajuste) if inglete else fuera <= d) & vv\n\n    def h(d):  # h(d) = rho(M(K\', d)) - d es estrictamente decreciente (Lema 2): una sola raiz\n        return regla(expandida(d), r, g0) - d\n\n    bajo = 0.0\n    if h(alto) > 0:  # no deberia pasar; si pasa, se usa la cota y se sigue\n        d = alto\n    else:\n        for _ in range(iteraciones):\n            medio = (bajo + alto) / 2\n            if h(medio) > 0:\n                bajo = medio\n            else:\n                alto = medio\n        d = (bajo + alto) / 2\n    salida = np.zeros(comp.shape, bool)\n    salida[y0:y1, x0:x1] = _refinar(c, d, expandida, r, g0)\n    return salida\n\n\ndef _refinar(c, d_bis, expandida, r, g0, ventana=(-2.5, 1.5), paso=0.25):\n    """Inversa en pixeles. La biseccion da un d continuo, pero la region expandida cambia a saltos (un anillo de\n    pixeles cada vez) y la region correcta puede quedar en el escalon de al lado del cruce (en un cuadrado de 32 px\n    con C1, la etiqueta usa D = 6,77 y la region original aparece para d en [6,0; 6,75), donde regla - d > 0).\n    Entre las regiones distintas que da d en la ventana [d - 2,5; d + 1,5] (pasos de 0,25) se elige la que,\n    encogida con la regla de la etiqueta, reproduce mejor el nucleo observado (IoU). Empate: la de d mas cercano\n    al cruce de la biseccion."""\n    vistas, mejor = set(), (-1.0, 0.0, None)\n    for d in np.arange(max(d_bis + ventana[0], 0.0), d_bis + ventana[1] + 1e-9, paso):\n        reg = expandida(float(d))\n        clave = (int(reg.sum()), hash(np.packbits(reg).tobytes()))\n        if clave in vistas or not reg.any():\n            continue\n        vistas.add(clave)\n        nuc = encoger(reg, regla(reg, r, g0))\n        inter = int((nuc & c).sum())\n        iou = inter / (int(nuc.sum()) + int(c.sum()) - inter)\n        if iou > mejor[0] + 1e-12 or (abs(iou - mejor[0]) <= 1e-12 and abs(d - d_bis) < mejor[1]):\n            mejor = (iou, abs(d - d_bis), reg)\n    return mejor[2] if mejor[2] is not None else expandida(d_bis)\n\n\n# ----------------------------------------------------------------------------- posproceso\ndef at_post(prob, v, umbral=0.3, puntaje_min=0.5, r=R, g0=G0, consistente=True, inglete=True, r_prima=R_PRIMA,\n            area_min=9, ajuste=None):\n    """Inferencia: binarizar P con 0,3, descartar componentes de area < 9 px o P media < 0,5 (como el codigo\n    oficial de DBNet) y expandir cada nucleo. consistente=True: con el d de C2. consistente=False: con r\' fijo,\n    como DBNet (D\' = A\' r\'/L\'), por distancia euclidea o, con inglete=True, en inglete (ablacion factorial)."""\n    etq, keep = componentes((prob > umbral) & v, area_min)\n    salida = np.zeros(prob.shape, bool)\n    for k in keep:\n        comp = etq == k\n        if prob[comp].mean() < puntaje_min:\n            continue\n        if consistente == "okr":\n            salida |= expandir(comp, distancia_okr(comp))\n        elif consistente:\n            salida |= expansion_consistente(comp, v, r, g0, inglete=inglete, ajuste=ajuste)\n        else:\n            A, L = area_perimetro(comp)\n            salida |= expandir_inglete(comp, A * r_prima / L, ajuste) if inglete else expandir(comp, A * r_prima / L)\n    return salida & v\n\n\ndef distancia_okr(comp):\n    """Distancia de dilatacion de OKR (Ma et al., 2023, ICASSP, doi 10.1109/ICASSP49357.2023.10094734, Ec. 2), el\n    trabajo mas cercano (experiments/novedad/lectura_okr.md): S = (C - sqrt(max(C^2 - 16 A, 0))) / 4, con C y A el\n    perimetro y el area de la region predicha. En un rectangulo a x b, C^2 - 16A = 4 (a - b)^2 y S = min(a, b): el\n    lado corto del nucleo. S es el TAMANO del elemento estructurante de la dilatacion (un diametro), igual que el\n    S de su erosion (Ec. 1, S = min lado / 2), asi que la distancia por lado es S / 2: el nucleo de ancho w/2 vuelve\n    a ancho w. Lectura declarada: el articulo redondea S a impar y no precisa la forma del elemento; aqui se usa\n    S / 2 sin redondeo y la expansion euclidea (expandir), para comparar solo la regla que elige la distancia contra\n    el punto fijo de C2."""\n    A, C = area_perimetro(comp)\n    return (C - np.sqrt(max(C * C - 16.0 * A, 0.0))) / 8.0\n\n\n# Variantes de la ablacion factorial 2^3 (diseno.md 5.1): nombre -> (g0, d consistente, inglete).\n# Ademas, una variante de comparacion con el trabajo mas cercano (protocolo §8): \'okr_sobre_at\' usa los nucleos del\n# brazo dbnet_at (C1) y la dilatacion de OKR en lugar de C2. Es exploratoria, fuera de la familia de la ablacion.\n# \'dbnet\' es el base: regla y posproceso de DBNet (g0 = 0, r\' fijo, expansion euclidea).\nVARIANTES = {\n    "dbnet": (0.0, False, False),\n    "c1_rfijo": (G0, False, False),        # solo encogimiento atenuado, con r\' = 1,5\n    "inglete_rfijo": (0.0, False, True),   # regla de DBNet, r\' = 1,5, expansion en inglete (C2b solo)\n    "c1_inglete_rfijo": (G0, False, True),  # C1 con r\' = 1,5 en inglete (sin C2a)\n    "c2_redondo": (0.0, True, False),      # regla de DBNet, d consistente, expansion euclidea\n    "c2": (0.0, True, True),               # regla de DBNet, d consistente, expansion en inglete\n    "at_redondo": (G0, True, False),       # C1 + d consistente con expansion euclidea\n    "dbnet_at": (G0, True, True),          # el modelo propio: C1 + C2\n    "okr_sobre_at": (G0, "okr", False),    # nucleos C1 + dilatacion de OKR (comparacion con el trabajo mas cercano)\n}\n\n\ndef objetivos_de(variante):\n    """Funcion (y, v) -> objetivos de entrenamiento de la variante. Solo importa g0 (C1)."""\n    g0 = VARIANTES[variante][0]\n    return lambda y, v: at_objetivos(y, v, g0=g0)\n\n\ndef post_de(variante):\n    """Funcion (P, v) -> mascara predicha de la variante (posproceso con o sin C2, con o sin inglete)."""\n    g0, cons, ing = VARIANTES[variante]\n    return lambda p, v: at_post(p, v, g0=g0, consistente=cons, inglete=ing)\n\n\n# DBNet base, con los nombres de E2\ndef db_objetivos(y, v, r=R, tmin=0.3, tmax=0.7):\n    return at_objetivos(y, v, r=r, g0=0.0, tmin=tmin, tmax=tmax)\n\n\ndef db_post(prob, v, **kw):\n    return at_post(prob, v, g0=0.0, consistente=False, inglete=False, **kw)\n', 'src/original_model.py': '"""UNetAT: la U-Net compacta del benchmark E2 con el mecanismo de DBNet-AT. Modelo original del Entregable 3.\n\nQue es. Se porta el mecanismo de DBNet (Liao et al., 2020, AAAI, "Real-time Scene Text Detection with\nDifferentiable Binarization") y el de DBNet-AT (entregable_3/diseno.md, seccion 3) a la U-Net compacta de E2\n(modelos_base.SmallUNet: tres niveles de 16, 32 y 64 canales, sin BatchNorm). Se entrena desde cero, sin pesos\npreentrenados, en el mismo pipeline de E2.\n\nArquitectura (la de SmallUNet, reescrita aqui, mas dos cabezas):\n    e1 = bloque(3 -> 16)             resolucion completa\n    e2 = bloque(16 -> 32)            1/2\n    e3 = bloque(32 -> 64)            1/4   (el mapa mas profundo: su paso, 4 px, es g0 de C1)\n    d2 = bloque(96 -> 32)            sube e3 y concatena e2\n    d1 = bloque(48 -> 16)            sube d2 y concatena e1\n    cabeza_prob, cabeza_umbral: dos cabezas sobre los 16 canales de d1 que dan los logits de P y de T.\nbloque(a, b) = Conv3x3(a,b) ReLU Conv3x3(b,b) ReLU. Las capas del codificador y del decodificador se crean en\nel mismo orden que en SmallUNet, asi que con la misma semilla global tienen los mismos pesos iniciales que la\nU-Net de E2; las cabezas se crean al final.\n\nFormulacion (DBNet, seccion 3 del articulo):\n    P = sigmoid(z_P)                       mapa de probabilidad del nucleo\n    T = sigmoid(z_T)                       mapa de umbral aprendido\n    B = sigmoid(k (P - T)),  k = 50        binarizacion diferenciable (aproxima el escalon H(P - T))\nDerivada de la binarizacion, la que hace entrenable al umbral:\n    dB/dP =  k B (1 - B)\n    dB/dT = -k B (1 - B)\nLa derivada es grande solo donde B ~ 0,5, es decir P ~ T: ahi el gradiente de la perdida empuja a P y a T a\nsepararse. Lejos del cruce B esta saturado y casi no hay gradiente.\n\nPerdida (perdida_db), la del codigo oficial de DBNet y la de E2 (entregable_2/codigo/modelos.py, DBNet.perdida):\n    L = 5 Ls + 10 Lt + Lb\n    Ls = BCE de P con minado de negativos dificiles 1:3 (OHEM)\n    Lt = L1 entre T y su blanco, promediada dentro de la banda Gd del mapa de umbral\n    Lb = Dice entre B y el nucleo\nLos pesos 5, 10 y 1 coinciden con diseno.md 3.1 ("5 Ls + 10 Lt + Lb"). El articulo escribe BCE para Lb con alfa = 1;\nel codigo oficial usa Dice y pesa 5 la BCE, y E2 sigue el codigo.\n\nLas etiquetas (nucleo, banda y blanco de T) las produce src.geometria_at.at_objetivos (C1 si variante=\'dbnet_at\',\nregla de DBNet si variante=\'dbnet\'). El posproceso (binarizar P, filtrar, expandir con C2) es predict_mask.\n"""\nfrom contextlib import nullcontext\n\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\nfrom . import geometria_at as G\n\nK_BINARIZACION = 50.0\n\n\ndef bloque(a, b):\n    """Dos convoluciones 3x3 con ReLU (el bloque de SmallUNet)."""\n    return nn.Sequential(nn.Conv2d(a, b, 3, padding=1), nn.ReLU(), nn.Conv2d(b, b, 3, padding=1), nn.ReLU())\n\n\nclass UNetAT(nn.Module):\n    """U-Net compacta con cabezas de probabilidad (P) y de umbral (T) y binarizacion B = sigmoid(k (P - T)).\n\n    Parametros:\n        k: pendiente de la binarizacion aproximada (50, como DBNet).\n        cabeza: \'1x1\' (una convolucion 1x1 por cabeza: 17 parametros cada una, paridad de capacidad con la U-Net\n            de E2) o \'3x3\' (conv 3x3 de 16 canales, ReLU y conv 1x1: mas capacidad en cada cabeza).\n        semilla: si se da, los pesos iniciales salen de esa semilla sin tocar el generador global; si es None se\n            usa el generador global (el arnes de E2 lo siembra con setup_seed antes de crear el modelo).\n    forward: en entrenamiento devuelve un dict con \'prob\' (P), \'thresh\' (T), \'binary\' (B) y \'logit_prob\' (z_P, que\n        la perdida necesita para la BCE estable); en eval devuelve solo P, de forma (B, 1, H, W).\n    """\n\n    def __init__(self, k=K_BINARIZACION, cabeza="1x1", semilla=None):\n        super().__init__()\n        if cabeza not in ("1x1", "3x3"):\n            raise ValueError("cabeza debe ser \'1x1\' o \'3x3\'")\n        self.k = float(k)\n        self.cabeza_tipo = cabeza\n        # fork_rng(devices=[]) guarda y restaura el generador de la CPU: crear el modelo con semilla no altera el global\n        contexto = torch.random.fork_rng(devices=[]) if semilla is not None else nullcontext()\n        with contexto:\n            if semilla is not None:\n                torch.manual_seed(int(semilla))\n            # mismo orden de creacion que SmallUNet: e1, e2, e3, d2, d1\n            self.e1, self.e2, self.e3 = bloque(3, 16), bloque(16, 32), bloque(32, 64)\n            self.d2, self.d1 = bloque(96, 32), bloque(48, 16)\n            self.cabeza_prob = self._cabeza(cabeza)\n            self.cabeza_umbral = self._cabeza(cabeza)\n\n    @staticmethod\n    def _cabeza(tipo):\n        if tipo == "1x1":\n            return nn.Conv2d(16, 1, 1)\n        return nn.Sequential(nn.Conv2d(16, 16, 3, padding=1), nn.ReLU(), nn.Conv2d(16, 1, 1))\n\n    # ------------------------------------------------------------------ validacion de la entrada\n    @staticmethod\n    def _validar(x):\n        if not isinstance(x, torch.Tensor):\n            raise ValueError(f"la entrada debe ser un tensor, no {type(x).__name__}")\n        if x.dim() != 4:\n            raise ValueError(f"se espera (B, 3, H, W); llego {tuple(x.shape)}")\n        if x.shape[1] != 3:\n            raise ValueError(f"se esperan 3 canales; llegaron {x.shape[1]}")\n        if not x.is_floating_point():\n            raise ValueError(f"se espera un tensor flotante, no {x.dtype}")\n        h, w = x.shape[-2:]\n        if h % 4 or w % 4:\n            raise ValueError(f"alto y ancho deben ser multiplos de 4 (dos reducciones a la mitad); llego {h} x {w}")\n\n    # ------------------------------------------------------------------ red\n    def _caracteristicas(self, x):\n        a = self.e1(x)\n        b = self.e2(F.max_pool2d(a, 2))\n        c = self.e3(F.max_pool2d(b, 2))\n        d = self.d2(torch.cat([F.interpolate(c, size=b.shape[-2:], mode="nearest"), b], 1))\n        return self.d1(torch.cat([F.interpolate(d, size=a.shape[-2:], mode="nearest"), a], 1))\n\n    def forward(self, x):\n        self._validar(x)\n        h = self._caracteristicas(x)\n        zp = self.cabeza_prob(h)\n        if not self.training:\n            return torch.sigmoid(zp)\n        p, t = torch.sigmoid(zp), torch.sigmoid(self.cabeza_umbral(h))\n        return {"prob": p, "thresh": t, "binary": torch.sigmoid(self.k * (p - t)), "logit_prob": zp}\n\n    # ------------------------------------------------------------------ inferencia\n    @torch.no_grad()\n    def predict_proba(self, x):\n        """P (B, 1, H, W), sin importar el modo del modulo."""\n        self._validar(x)\n        return torch.sigmoid(self.cabeza_prob(self._caracteristicas(x)))\n\n    @torch.no_grad()\n    def predict_mask(self, x, variante="dbnet_at", valida=None):\n        """Mascara binaria (B, H, W) bool: P pasa por el posproceso de la variante (src.geometria_at.VARIANTES).\n        \'dbnet_at\' = C1 + C2 (expansion consistente en inglete); \'dbnet\' = el posproceso de DBNet (r\' = 1,5).\n        valida: bool (B, H, W) con la zona valida (sin relleno del letterbox); por defecto, toda la imagen."""\n        if variante not in G.VARIANTES:\n            raise ValueError(f"variante desconocida: {variante}; opciones: {sorted(G.VARIANTES)}")\n        p = self.predict_proba(x)[:, 0].cpu().numpy()\n        v = np.ones(p.shape, bool) if valida is None else np.asarray(valida, bool).reshape(p.shape)\n        post = G.post_de(variante)\n        return np.stack([post(p[i], v[i]) for i in range(len(p))])\n\n    def perdida(self, salida, objetivos, **pesos):\n        return perdida_db(salida, objetivos, **pesos)\n\n\n# ---------------------------------------------------------------------- perdida (copia de la de E2)\ndef bce_ohem(logits, gt, mask, razon=3):\n    """BCE con minado de negativos dificiles 1:razon (E2, modelos.bce_ohem; codigo oficial de DBNet)."""\n    perd = F.binary_cross_entropy_with_logits(logits, gt, reduction="none")\n    pos = (gt > 0.5) & (mask > 0.5)\n    neg = (gt <= 0.5) & (mask > 0.5)\n    n_pos = int(pos.sum())\n    # sin positivos (imagen sin rotulo) se toman 100 negativos, como en las implementaciones oficiales\n    n_neg = min(int(neg.sum()), razon * n_pos if n_pos else 100)\n    perd_neg = torch.topk(perd[neg], n_neg).values if n_neg else perd.new_zeros(0)\n    return (perd[pos].sum() + perd_neg.sum()) / (n_pos + n_neg + 1e-6)\n\n\ndef dice_perdida(p, t, m):\n    """1 - Dice enmascarado, con p, t y m del mismo tamano (E2, modelos.dice_perdida)."""\n    p, t = p * m, t * m\n    return 1 - (2 * (p * t).sum() + 1e-6) / ((p * p).sum() + (t * t).sum() + 1e-6)\n\n\ndef perdida_db(salida, objetivos, peso_s=5.0, peso_t=10.0, peso_b=1.0):\n    """L = 5 Ls + 10 Lt + Lb, la de DBNet segun el codigo oficial y E2.\n\n    salida: dict de UNetAT en modo entrenamiento. objetivos: dict de tensores (B, 1, H, W) con las claves de\n    geometria_at.at_objetivos (db_gt, db_mask, db_umbral, db_umbral_mask)."""\n    ls = bce_ohem(salida["logit_prob"], objetivos["db_gt"], objetivos["db_mask"])\n    lb = dice_perdida(salida["binary"], objetivos["db_gt"], objetivos["db_mask"])\n    m = objetivos["db_umbral_mask"]\n    lt = (torch.abs(salida["thresh"] - objetivos["db_umbral"]) * m).sum() / (m.sum() + 1e-6)\n    return peso_s * ls + peso_t * lt + peso_b * lb\n', 'training/entrenar_original.py': '"""Entrena UNetAT (el modelo original de E3) con el arnes del benchmark E2.\n\nReutiliza, importandolos por ruta desde el repositorio de E2 (solo lectura), el cargador de datos y los\nutilidades del arnes: dataset_ready (partition, load_example, metrics) y experimento_e2_colab (setup_seed,\ntraining_batches, make_optimizer, rng_state, restore_rng, atomic_json, atomic_torch, sha, digest, environment,\nverify_sources). Lo que NO se puede reutilizar tal cual y se reescribe aqui, igual en lo esencial:\n- train_model: construye el modelo desde MODELS_E2 y calcula la perdida con compute_loss, que no conocen a UNetAT.\n  Aqui es el mismo bucle (misma semilla y orden de lotes por epoca, mismo optimizador, mismas epocas y paciencia,\n  mismos checkpoints) con perdida_db sobre las etiquetas de DBNet o de DBNet-AT.\n- evaluate_validation: calculaba el Dice con P >= 0,5. Aqui la mascara sale del posproceso de la variante\n  (para dbnet_at, C2) y se promedia por grupo como alla; mismo fork_rng con la semilla 20261118.\n- verify_data y el preflight de E2: leen las imagenes desde repo_dl_imo/e2/VOC2012, que no existe en el\n  repositorio de solo lectura; aqui se verifican los mismos SHA-256 de imagenes y mascaras contra los CSV, leyendo\n  las imagenes desde --voc. El preflight de E2 (preflight.json) solo cubre sus 10 modelos; aqui hay uno propio:\n  un forward y backward con perdida finita antes de entrenar, anotado en preflight_original.json.\n\nReceta de la U-Net compacta de E2 (config/experimento_e2.json, modelos.unet): Adam, lr 1e-3, weight_decay 0, lote 2,\n30 epocas, paciencia 5, entrada 256, sin preentrenamiento, sin aumentos (load_example no los hace), seleccion por el\nDice medio por grupo en validacion; aqui, despues del posproceso.\n\nUso en GPU (Colab):\n    python entrenar_original.py --variante dbnet dbnet_at --seeds 11 23 37 53 71 --output /content/drive/MyDrive/e3_original\nHumo en CPU:\n    python entrenar_original.py --cpu-smoke --allow-local-output --max-images 8 --epochs 2 --seeds 11 \\\n        --variante dbnet_at --output C:\\\\Proyectos\\\\estudio\\\\datos\\\\deep_learning\\\\e3_humo\nLas otras 6 variantes de la ablacion 2^3 no se entrenan: salen por inferencia con los mismos checkpoints\n(geometria_at.VARIANTES).\n"""\nimport argparse\nimport copy\nimport hashlib\nimport json\nimport sys\nimport time\nfrom collections import defaultdict\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\n\nE3 = Path(__file__).resolve().parents[1]\nE2_SCRIPTS = Path(r"C:\\Proyectos\\estudio\\datos\\deep_learning\\repo_dl_imo\\e2\\tesis_imo_para_codex\\scripts")\nVOC = Path(r"C:\\Proyectos\\estudio\\datos\\hazmat13_drive")\nSEMILLA_VALIDACION = 20261118  # la de evaluate_validation de E2\nCLAVES_BINARIAS = ("db_gt", "db_mask", "db_umbral_mask")  # 0/1: se guardan en uint8\nH = DS = G = UNetAT = perdida_db = None  # se importan en cargar_arnes()\n\n\ndef cargar_arnes(e2_scripts):\n    """Importa el arnes de E2 por ruta y los modulos de E3. Falla con un mensaje claro si falta algo."""\n    global H, DS, G, UNetAT, perdida_db\n    e2_scripts = Path(e2_scripts).resolve()\n    if not (e2_scripts / "experimento_e2_colab.py").exists():\n        raise FileNotFoundError(f"No esta el arnes de E2 en {e2_scripts}")\n    for ruta in (str(e2_scripts), str(E3)):\n        if ruta not in sys.path:\n            sys.path.insert(0, ruta)\n    import experimento_e2_colab as h\n    import dataset_ready as ds\n    from src import geometria_at as g\n    from src.original_model import UNetAT as red, perdida_db as perdida\n    H, DS, G, UNetAT, perdida_db = h, ds, g, red, perdida\n    return e2_scripts\n\n\n# ------------------------------------------------------------------------------------------ datos y trazabilidad\ndef firma_codigo(e2_scripts):\n    """SHA-256 conjunto del codigo de E3 (src y este script) y de los archivos del arnes de E2 que se usan."""\n    e3 = {str(p.relative_to(E3)): H.sha(p) for p in sorted((E3 / "src").glob("*.py"))}\n    e3["training/entrenar_original.py"] = H.sha(Path(__file__))\n    e2 = {n: H.sha(Path(e2_scripts) / n) for n in\n          ("experimento_e2_colab.py", "dataset_ready.py", "modelos_base.py", "preparar_base.py", "inicio_voc.py")}\n    return {"e3": H.digest(e3), "e2_arnes": H.digest(e2)}\n\n\ndef verificar_datos(filas, voc):\n    """Lo que hace verify_data de E2, con las imagenes leidas desde voc: sin fuga de grupos entre particiones\n    (del test solo se leen los metadatos del CSV, como E2), SHA-256 de imagen y mascara de cada fila usada y\n    normalizacion a 256. Devuelve el resumen de hashes de los CSV y de las filas usadas."""\n    base = H.PROJECT / "resultados/preentrenamiento"\n    firma = {f: H.sha(base / f) for f in ["train.csv", "val.csv", "test.csv", "normalization_train.json"]}\n    grupos = {s: {r["group_id"] for r in DS.partition(s, allow_test=(s == "test"))} for s in ("train", "val", "test")}\n    for a, b in (("train", "val"), ("train", "test"), ("val", "test")):\n        if grupos[a] & grupos[b]:\n            raise RuntimeError(f"Fuga de grupos entre {a}/{b}")\n    usados = {}\n    for particion, rows in filas.items():\n        if not rows:\n            raise RuntimeError(f"Particion vacia: {particion}")\n        for row in rows:\n            for ruta, esperado in ((Path(voc) / row["image_path"], row["image_sha256"]),\n                                   (H.PROJECT / row["binary_mask_path"], row["binary_mask_sha256"])):\n                if H.sha(ruta) != esperado:\n                    raise RuntimeError(f"Datos modificados: {ruta}")\n            usados[row["id"]] = row["image_sha256"]\n    norma = json.loads((base / "normalization_train.json").read_text())\n    if norma["size"] != 256:\n        raise RuntimeError("Normalizacion no corresponde a 256")\n    return H.digest({"csv": firma, "filas_usadas": usados})\n\n\nclass Datos:\n    """Imagenes de una particion cargadas en RAM (load_example no hace aumentos: es determinista) y, para train,\n    los objetivos de cada variante, calculados una vez por imagen."""\n\n    def __init__(self, filas, voc, objetivos=None, nombre=""):\n        self.filas, self.x, self.y, self.v, self.obj = filas, {}, {}, {}, {}\n        t0 = time.perf_counter()\n        for fila in filas:\n            x, y, v = DS.load_example(fila, data_root=voc)\n            self.x[fila["id"]], self.y[fila["id"]], self.v[fila["id"]] = x, y, v\n            if objetivos is not None:\n                o = objetivos(y, v)\n                self.obj[fila["id"]] = {k: (a.astype(np.uint8) if k in CLAVES_BINARIAS else a) for k, a in o.items()}\n        print(f"[datos] {nombre}: {len(filas)} imagenes en RAM ({time.perf_counter() - t0:.1f} s)", flush=True)\n\n    def lote(self, filas, device):\n        """(x, objetivos) como tensores. x: (B, 3, H, W); cada objetivo: (B, 1, H, W) float32."""\n        ids = [f["id"] for f in filas]\n        x = torch.from_numpy(np.stack([self.x[i] for i in ids])).to(device)\n        obj = {k: torch.from_numpy(np.stack([self.obj[i][k] for i in ids]).astype(np.float32)).to(device)\n               for k in self.obj[ids[0]]}\n        return x, obj\n\n\n# ------------------------------------------------------------------------------------------ validacion\n@torch.no_grad()\ndef evaluar_validacion(modelo, datos, variante, device):\n    """Como evaluate_validation de E2 (Dice por imagen, promedio por grupo, luego promedio de los grupos), pero la\n    mascara sale de predict_mask (posproceso de la variante) en lugar de P >= 0,5."""\n    modelo.eval()\n    agrupado, detalle = defaultdict(list), []\n    dispositivos = [torch.cuda.current_device()] if device.type == "cuda" else []\n    with torch.random.fork_rng(devices=dispositivos):\n        torch.manual_seed(SEMILLA_VALIDACION)\n        for fila in datos.filas:\n            i = fila["id"]\n            x = torch.from_numpy(datos.x[i][None]).to(device)\n            mascara = modelo.predict_mask(x, variante, valida=datos.v[i][None])[0]\n            puntajes = DS.metrics(mascara.astype(np.float32), datos.y[i], datos.v[i])\n            agrupado[fila["group_id"]].append(puntajes["Dice"])\n            detalle.append({"id": i, "group_id": fila["group_id"], **puntajes})\n    return float(np.mean([np.mean(v) for v in agrupado.values()])), detalle\n\n\n# ------------------------------------------------------------------------------------------ entrenamiento\ndef preflight(variante, config, identity, train, device, destino):\n    """Un forward y backward con perdida y gradientes finitos antes de entrenar (sustituye al preflight de E2)."""\n    H.setup_seed(0)\n    modelo = UNetAT().to(device).train()\n    x, obj = train.lote(train.filas[:2], device)\n    perdida = perdida_db(modelo(x), obj)\n    perdida.backward()\n    ok = bool(torch.isfinite(perdida)) and all(p.grad is not None and bool(torch.isfinite(p.grad).all())\n                                                for p in modelo.parameters())\n    informe = {"variante": variante, "status": "passed" if ok else "failed", "perdida": float(perdida.detach()),\n               "identity": identity, "config_hash": H.digest(config), "environment": H.environment()}\n    H.atomic_json(Path(destino) / f"preflight_original_{variante}.json", informe)\n    if not ok:\n        raise RuntimeError(f"Preflight fallido ({variante}): perdida {float(perdida.detach())}")\n\n\ndef entrenar(variante, semilla, output, config, identity, device, train, val, parada=None):\n    """train_model de E2 con UNetAT. Mismo formato de config.json, last.pt, best_epoch_XXX.pt, history.json y\n    result.json (fingerprint, best_file, last_sha256, best_sha256)."""\n    nombre = f"unetat_{variante}"\n    H.setup_seed(semilla)\n    dest = Path(output) / f"{nombre}_seed{semilla}"\n    dest.mkdir(parents=True, exist_ok=True)\n    spec = {"name": nombre, "seed": semilla, "config": config, "identity": identity, "environment": H.environment(),\n            "modelo": {"k": 50.0, "cabeza": "1x1", "g0": G.G0 if variante == "dbnet_at" else 0.0,\n                       "perdida": "5*Ls + 10*Lt + Lb"}}\n    huella = H.digest(spec)\n    cfg_file = dest / "config.json"\n    if cfg_file.exists():\n        if json.loads(cfg_file.read_text())["fingerprint"] != huella:\n            raise RuntimeError(f"Corrida incompatible: {dest}. Usa una carpeta de experimento nueva.")\n    else:\n        H.atomic_json(cfg_file, {"fingerprint": huella, **spec})\n    last = dest / "last.pt"\n    if (dest / "result.json").exists():\n        res = json.loads((dest / "result.json").read_text())\n        if res.get("fingerprint") != huella or not res.get("completed"):\n            raise RuntimeError("Resultado incompatible")\n        if not last.exists() or not (dest / res["best_file"]).exists():\n            raise RuntimeError("Faltan checkpoints de corrida completada")\n        if H.sha(last) != res["last_sha256"] or H.sha(dest / res["best_file"]) != res["best_sha256"]:\n            raise RuntimeError("Checkpoint completado corrupto/modificado")\n        return res\n    modelo = UNetAT().to(device)  # tras setup_seed: mismos pesos iniciales de codificador/decodificador que la U-Net de E2\n    optimizador = H.make_optimizer(modelo, config["models"]["unet"])\n    inicio, mejor, sin_mejora, historia, mejor_archivo = 0, -1.0, 0, [], None\n    if last.exists():\n        guardado = torch.load(last, map_location="cpu", weights_only=False)\n        if guardado["fingerprint"] != huella:\n            raise RuntimeError("Checkpoint incompatible")\n        modelo.load_state_dict(guardado["model"])\n        optimizador.load_state_dict(guardado["optimizer"])\n        inicio, mejor, sin_mejora, historia, mejor_archivo = (guardado["epoch"] + 1, guardado["best"], guardado["stale"],\n                                                              guardado["history"], guardado["best_file"])\n        if not (dest / mejor_archivo).exists():\n            raise RuntimeError("Falta mejor checkpoint referenciado")\n        H.restore_rng(guardado["rng"])\n    if parada is not None and inicio >= parada and inicio < config["epochs"] and sin_mejora < config["patience"]:\n        return {"completed": False, "paused": True, "epochs_completed": inicio}\n    for epoca in range(inicio, config["epochs"]):\n        if sin_mejora >= config["patience"]:\n            break\n        modelo.train()\n        t0, suma, vistas = time.perf_counter(), 0.0, 0\n        for filas in H.training_batches(train.filas, semilla, epoca, config["batch_size"]):\n            x, obj = train.lote(filas, device)\n            optimizador.zero_grad(set_to_none=True)\n            perdida = perdida_db(modelo(x), obj)\n            if not torch.isfinite(perdida):\n                raise RuntimeError("Perdida no finita")\n            perdida.backward()\n            optimizador.step()\n            suma += perdida.item() * len(filas)\n            vistas += len(filas)\n        t_entreno = time.perf_counter() - t0\n        puntaje, detalle = evaluar_validacion(modelo, val, variante, device)\n        if not np.isfinite(puntaje):\n            raise RuntimeError("Validacion no finita")\n        if puntaje > mejor:\n            mejor, sin_mejora = puntaje, 0\n            mejor_archivo = f"best_epoch_{epoca + 1:03d}.pt"\n            H.atomic_torch(dest / mejor_archivo, {"model": modelo.state_dict(), "epoch": epoca, "fingerprint": huella,\n                                                  "validation": detalle})\n        else:\n            sin_mejora += 1\n        historia.append({"epoch": epoca + 1, "train_loss": suma / vistas, "val_group_Dice": puntaje,\n                         "seconds": time.perf_counter() - t0, "seconds_train": t_entreno})\n        # last apunta a un best inmutable ya escrito: un corte no cambia su significado\n        H.atomic_torch(last, {"fingerprint": huella, "epoch": epoca, "model": modelo.state_dict(),\n                              "optimizer": optimizador.state_dict(), "best": mejor, "stale": sin_mejora,\n                              "history": historia, "best_file": mejor_archivo, "rng": H.rng_state()})\n        H.atomic_json(dest / "history.json", historia)\n        for viejo in dest.glob("best_epoch_*.pt"):\n            if viejo.name != mejor_archivo:\n                viejo.unlink()\n        print(f"{nombre} seed={semilla} epoch={epoca + 1} loss={suma / vistas:.4f} val={puntaje:.5f} "\n              f"({historia[-1][\'seconds\']:.1f} s; entrenamiento {t_entreno:.1f} s)", flush=True)\n        if (parada is not None and epoca + 1 >= parada\n                and epoca + 1 < config["epochs"] and sin_mejora < config["patience"]):\n            return {"completed": False, "paused": True, "epochs_completed": epoca + 1}\n    res = {"completed": True, "fingerprint": huella, "model": nombre, "variante": variante, "seed": semilla,\n           "best_val_group_Dice": mejor, "epochs_completed": len(historia), "best_file": mejor_archivo,\n           "parameters": sum(p.numel() for p in modelo.parameters()),\n           "last_sha256": H.sha(last), "best_sha256": H.sha(dest / mejor_archivo)}\n    H.atomic_json(dest / "result.json", res)\n    return res\n\n\ndef main(argv=None):\n    ap = argparse.ArgumentParser(description=__doc__.split("\\n")[0])\n    ap.add_argument("--variante", nargs="+", default=["dbnet", "dbnet_at"], choices=["dbnet", "dbnet_at"],\n                    help="brazos que se entrenan; las otras 6 variantes salen por inferencia")\n    ap.add_argument("--seeds", nargs="+", type=int, help="por defecto, las del config de E2")\n    ap.add_argument("--output", required=True)\n    ap.add_argument("--config", default=None, help="por defecto, config/experimento_e2.json del arnes de E2")\n    ap.add_argument("--allow-local-output", action="store_true")\n    ap.add_argument("--cpu-smoke", action="store_true", help="humo en CPU; exige --max-images")\n    ap.add_argument("--max-images", type=int, help="usa solo las primeras N imagenes de train y de val (humo)")\n    ap.add_argument("--epochs", type=int, help="sobrescribe las epocas del config (humo)")\n    ap.add_argument("--pause-after-epoch", type=int,\n                    help="pausa después de esta época absoluta sin cambiar la receta; omitir para reanudar")\n    ap.add_argument("--voc", default=str(VOC), help="carpeta que contiene VOC2012/ (las imagenes)")\n    ap.add_argument("--e2-scripts", default=str(E2_SCRIPTS))\n    args = ap.parse_args(argv)\n    if args.pause_after_epoch is not None and args.pause_after_epoch < 1:\n        ap.error("--pause-after-epoch debe ser positivo")\n\n    e2_scripts = cargar_arnes(args.e2_scripts)\n    output = Path(args.output).resolve()\n    if not args.allow_local_output:\n        import subprocess\n        subprocess.run(["mountpoint", "-q", "/content/drive"], check=True)\n        if not output.is_relative_to(Path("/content/drive/MyDrive")):\n            raise RuntimeError("Salida debe estar en Drive")\n    if args.cpu_smoke:\n        if not args.max_images or args.max_images > 64:\n            raise RuntimeError("--cpu-smoke exige --max-images (hasta 64): una corrida completa en CPU no es un humo")\n        device = torch.device("cpu")\n    else:\n        if not torch.cuda.is_available():\n            raise RuntimeError("Este entrenamiento requiere GPU; para un humo en CPU usa --cpu-smoke --max-images N")\n        device = torch.device("cuda")\n        if args.max_images or args.epochs:\n            raise RuntimeError("--max-images y --epochs son solo para --cpu-smoke")\n    config = json.loads(Path(args.config or H.PROJECT / "config/experimento_e2.json").read_text())\n    if (config["pretrained"] or config["size"] != 256 or config["batch_size"] < 2 or config["threshold"] != .5\n            or config["epochs"] < 1 or config["patience"] < 1\n            or config["loss"] != "0.5_BCEWithLogits_plus_0.5_softDice_per_image_valid"):\n        raise ValueError("Configuracion no soportada")\n    config = copy.deepcopy(config)\n    if args.epochs:\n        config["epochs"] = args.epochs\n    if args.max_images:\n        config["max_images"] = args.max_images  # entra en la huella: un humo nunca se confunde con una corrida completa\n\n    H.verify_sources()\n    train_filas, val_filas = DS.partition("train"), DS.partition("val")\n    if args.max_images:\n        train_filas, val_filas = train_filas[:args.max_images], val_filas[:args.max_images]\n    identity = {**firma_codigo(e2_scripts),\n                "data": verificar_datos({"train": train_filas, "val": val_filas}, args.voc)}\n    output.mkdir(parents=True, exist_ok=True)\n    val = Datos(val_filas, args.voc, nombre="val")\n    resultados = []\n    for variante in args.variante:\n        train = Datos(train_filas, args.voc, G.objetivos_de(variante), nombre=f"train ({variante}, objetivos C1)"\n                      if variante == "dbnet_at" else f"train ({variante}, objetivos DBNet)")\n        preflight(variante, config, identity, train, device, output)\n        for semilla in args.seeds or config["seeds"]:\n            resultados.append(entrenar(variante, semilla, output, config, identity, device, train, val,\n                                       parada=args.pause_after_epoch))\n    print(json.dumps(resultados, indent=1))\n    return resultados\n\n\nif __name__ == "__main__":\n    main()\n'}
FIRMAS = {'src/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'src/geometria_at.py': '7dba12fbba9333b04c4ce1ae96f305cdcb7c0865337d722b8c77f3538fd4676c', 'src/original_model.py': '395340738602521a528e3c14cc662e0cb3f737954d1b055a9e82c0b605fdcbff', 'training/entrenar_original.py': '3d69820ef6ef3a8167d052cad85b71c926b9f0ec46ac2dee468b71ad12e6ff07'}
manifest = OUTPUT / 'firmas_codigo_e3.json'
if manifest.exists() and json.loads(manifest.read_text()) != FIRMAS:
    raise RuntimeError('Hay otro código en esta carpeta de salida; no mezclar corridas.')
for rel, texto in ARCHIVOS.items():
    p = E3 / rel
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(texto, encoding='utf-8')
    assert sha_file(p) == FIRMAS[rel]
manifest.write_text(json.dumps(FIRMAS, indent=2))
freeze = subprocess.check_output([sys.executable, '-m', 'uv', 'pip', 'freeze', '--python', PY], text=True)
(OUTPUT / 'environment.freeze.txt').write_text(freeze)
print('Código verificado. Destino nuevo:', OUTPUT)


In [ ]:
import time
inicio = time.perf_counter()
comando = [PY, str(E3 / 'training/entrenar_original.py'), '--output', str(OUTPUT),
           '--variante', 'dbnet', 'dbnet_at', '--seeds', '11',
           '--voc', str(BASE), '--e2-scripts', str(PROJECT / 'scripts')]
subprocess.run(comando + ['--pause-after-epoch', '2'], check=True)
duracion = time.perf_counter() - inicio


In [ ]:
import statistics
resumen = {'estado': 'piloto_pausado_no_resultado_final', 'segundos_celda': duracion,
           'test_evaluado': False, 'brazos': {}}
for brazo in ['dbnet', 'dbnet_at']:
    carpeta = OUTPUT / f'unetat_{brazo}_seed11'
    hist = json.loads((carpeta / 'history.json').read_text())
    assert len(hist) >= 2, 'Piloto incompleto'
    assert (carpeta / 'last.pt').is_file()
    promedio = statistics.mean(h['seconds'] for h in hist)
    resumen['brazos'][brazo] = {'epocas': len(hist), 'segundos_medios_epoca': promedio,
        'segundos_medios_entrenamiento': statistics.mean(h['seconds_train'] for h in hist),
        'dice_validacion_ultima_epoca': hist[-1]['val_group_Dice'],
        'checkpoint_sha256': sha_file(carpeta / 'last.pt')}
resumen['horas_proyectadas_20_corridas_30_epocas'] = sum(
    b['segundos_medios_epoca'] * 30 * 10 for b in resumen['brazos'].values()) / 3600
resumen['nota_proyeccion'] = ('Extrapolación de dos épocas, no promesa: excluye preparación, HPO, '
    'rivales y cambios de costo del posproceso; early stopping puede reducir el tiempo.')
(OUTPUT / 'resumen_piloto.json').write_text(json.dumps(resumen, indent=2, ensure_ascii=False))
print(json.dumps(resumen, indent=2, ensure_ascii=False))
from google.colab import files
files.download(str(OUTPUT / 'resumen_piloto.json'))


## Al terminar

Guarda `resumen_piloto.json` y anota las unidades de Colab antes y después. Desconecta el entorno GPU.
Los checkpoints quedan en Drive. Comparte el resumen para decidir el presupuesto de la campaña completa.

Para reanudar únicamente estos dos brazos de semilla 11, en la MISMA sesión y tras revisar el presupuesto,
agrega una celda con `subprocess.run(comando, check=True)`. Esto quita la pausa y conserva la receta.
Si cambió el entorno, el entrenador verifica su identidad y rechaza una reanudación incompatible.
No uses el cuaderno histórico `Original_E3.ipynb` para reanudar esta carpeta: tiene otro código y destino.
